# Listening check

Qualitative spot-check of retrieval: pick a query track, show its nearest
neighbors with tags, and play the query plus its top matches so you can judge
sonic similarity by ear. Run `audio-sim embed` first so the index exists.

Install notebook extras: `pip install -e .[notebook]`

In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable when running from notebooks/.
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
from IPython.display import Audio, display

from audio_similarity.config import Config
from audio_similarity.index.store import load_index
from audio_similarity.index.search import cosine_topk
from audio_similarity.data.metadata import load_autotagging_tsv

cfg = Config.load(Path.cwd().parent / 'config.yaml')
ids, vectors = load_index(Path.cwd().parent / cfg.paths.index_path)
tsv = Path.cwd().parent / cfg.paths.metadata_dir / 'autotagging.tsv'
tracks = load_autotagging_tsv(tsv) if tsv.exists() else {}
raw = Path.cwd().parent / cfg.paths.raw_audio_dir
print(f'{len(ids)} songs indexed, {len(tracks)} with metadata')

In [ ]:
def tags_str(key, category='genre'):
    if key not in tracks:
        return ''
    return ', '.join(t.split('---')[1] for t in sorted(tracks[key].tags_in(category)))

def audio_path(key):
    return str(raw / tracks[key].path) if key in tracks else None

def show_neighbors(query_key, k=5):
    qpos = ids.index(query_key)
    idx, scores = cosine_topk(vectors[qpos], vectors, k=k, exclude_index=qpos)
    print(f'QUERY  {query_key}  [{tags_str(query_key)}]')
    p = audio_path(query_key)
    if p and Path(p).exists():
        display(Audio(p))
    for rank, (j, s) in enumerate(zip(idx, scores), 1):
        key = ids[int(j)]
        print(f'  {rank}. {key}  cos={s:.3f}  [{tags_str(key)}]')
        ap = audio_path(key)
        if ap and Path(ap).exists():
            display(Audio(ap))

In [ ]:
# Pick the first indexed track, or set query_key to any id you want to inspect.
query_key = ids[0]
show_neighbors(query_key, k=5)